# Graph & Social Network Mining Demo: Embeddings and Link Prediction

This notebook introduces two important topics in **graph and social network mining**:

- **Graph embeddings**: representing nodes as low-dimensional vectors that preserve structural information
- **Link prediction**: estimating which missing or future edges are likely to appear in a network

These topics are widely used in:
- friend recommendation in social networks
- citation and collaboration prediction
- knowledge graph completion
- fraud and anomaly detection

The notebook combines theory and practice. We will build small networks, extract structural features, compute simple embeddings, compare node similarity in vector space, and perform link prediction using classical graph heuristics.

#### Import required packages

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

print("Libraries loaded.")
print("NetworkX version:", nx.__version__)

## Create an Example Social Network

We start with a small undirected graph representing friendship or interaction ties.  
The network is designed to show local groups, bridge nodes, and plausible missing connections.

In [ ]:
G = nx.Graph()

edges = [
    ("Alice", "Bob"), ("Alice", "Carol"), ("Alice", "David"),
    ("Bob", "Carol"), ("Bob", "Eve"),
    ("Carol", "David"), ("Carol", "Eve"),
    ("David", "Frank"),
    ("Eve", "Grace"), ("Eve", "Heidi"),
    ("Frank", "Grace"), ("Frank", "Ivan"),
    ("Grace", "Heidi"), ("Grace", "Ivan"),
    ("Heidi", "Judy"),
    ("Ivan", "Judy"), ("Ivan", "Karl"),
    ("Judy", "Karl"),
]

G.add_edges_from(edges)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print(sorted(G.nodes()))

## Visualize the Network

A visualization helps us inspect clusters, bridges, and likely missing links.  
In a real social network, users inside the same local region often have a higher chance of connecting.

In [ ]:
plt.figure(figsize=(9, 6))
pos = nx.spring_layout(G, seed=7)
nx.draw(G, pos, with_labels=True, node_size=1400, font_size=9)
plt.title("Example Social Network")
plt.axis("off")
plt.show()

## Inspect Basic Neighborhood Structure

Before building embeddings, it is useful to look at the adjacency relationships and degree distribution.  
This gives intuition about local density and network roles.

In [ ]:
adj_df = nx.to_pandas_adjacency(G, dtype=int)
degree_df = pd.DataFrame({
    "Node": list(dict(G.degree()).keys()),
    "Degree": list(dict(G.degree()).values())
}).sort_values("Degree", ascending=False)

display(adj_df)
display(degree_df)

## Build an Adjacency Matrix for Embedding

A simple starting point is the **adjacency matrix**, where each row encodes direct connectivity.  
Although high-dimensional, this matrix can be reduced into a smaller vector representation.

In [ ]:
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
nodes = sorted(G.nodes())

print("Adjacency matrix shape:", A.shape)
pd.DataFrame(A, index=nodes, columns=nodes)

## Create a Simple Graph Embedding with Truncated SVD

One classical idea is to apply a low-rank matrix factorization to the adjacency matrix.  
This is not the only embedding method, but it is easy to explain and works well for classroom demonstration.

Interpretation:
- each node becomes a low-dimensional vector
- structurally similar nodes may appear close in the new vector space

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=42)
X_emb = svd.fit_transform(A)

emb_df = pd.DataFrame(X_emb, columns=["Emb1", "Emb2"])
emb_df.insert(0, "Node", nodes)

print("Explained variance ratio:", svd.explained_variance_ratio_)
emb_df

## Visualize Node Embeddings in 2D

Each point now represents a node in embedding space.  
Nodes with similar neighborhoods often appear closer together.

In [ ]:
plt.figure(figsize=(8, 6))
for i, node in enumerate(nodes):
    plt.scatter(X_emb[i, 0], X_emb[i, 1], s=100)
    plt.text(X_emb[i, 0] + 0.02, X_emb[i, 1] + 0.02, node, fontsize=9)

plt.title("2D Node Embeddings from Adjacency Matrix")
plt.xlabel("Embedding Dimension 1")
plt.ylabel("Embedding Dimension 2")
plt.grid(True)
plt.show()

## Compare Similarity Between Nodes in Embedding Space

A useful benefit of embeddings is that we can compute vector similarity directly.  
This supports tasks such as:
- nearest-neighbor search
- recommendation
- clustering
- candidate edge generation

In [ ]:
sim_matrix = cosine_similarity(X_emb)
sim_df = pd.DataFrame(sim_matrix, index=nodes, columns=nodes).round(3)
sim_df

## Find the Most Similar Nodes to a Target User

Suppose we want to recommend potential contacts for a specific user based on embedding similarity.  
We inspect the most similar nodes to **Alice** in the learned vector space.

In [ ]:
target = "Alice"
target_idx = nodes.index(target)

similarities = []
for i, node in enumerate(nodes):
    if node != target:
        similarities.append((node, sim_matrix[target_idx, i]))

similarity_rank_df = pd.DataFrame(similarities, columns=["Node", "CosineSimilarity"])
similarity_rank_df = similarity_rank_df.sort_values("CosineSimilarity", ascending=False)
similarity_rank_df

## Define the Link Prediction Task

For link prediction, we score **currently missing edges**.  
In an undirected graph, a candidate pair is any pair of nodes that does not already have an edge.

We first enumerate all missing pairs.

In [ ]:
all_pairs = list(combinations(nodes, 2))
missing_edges = [(u, v) for u, v in all_pairs if not G.has_edge(u, v)]

print("Number of candidate missing edges:", len(missing_edges))
missing_edges[:15]

## Common Neighbors Score

The **common neighbors** score counts how many mutual neighbors two nodes share.

This reflects a social-network principle known as **triadic closure**:
if two users share many friends, they are more likely to connect.

In [ ]:
def common_neighbors_score(graph, u, v):
    return len(list(nx.common_neighbors(graph, u, v)))

cn_scores = [(u, v, common_neighbors_score(G, u, v)) for u, v in missing_edges]
cn_df = pd.DataFrame(cn_scores, columns=["Node1", "Node2", "CommonNeighbors"])
cn_df.sort_values("CommonNeighbors", ascending=False).head(15)

## Jaccard Coefficient

The **Jaccard coefficient** normalizes shared neighbors by the total neighborhood size.

This helps when comparing pairs involving nodes with very different degree sizes.

In [ ]:
jaccard_scores = list(nx.jaccard_coefficient(G, missing_edges))
jaccard_df = pd.DataFrame(jaccard_scores, columns=["Node1", "Node2", "Jaccard"])
jaccard_df.sort_values("Jaccard", ascending=False).head(15)

## Adamic–Adar Index

The **Adamic–Adar** score gives more weight to shared neighbors that are themselves less connected.

Intuition:
- a rare shared neighbor may be more informative than a very popular shared neighbor

In [ ]:
aa_scores = list(nx.adamic_adar_index(G, missing_edges))
aa_df = pd.DataFrame(aa_scores, columns=["Node1", "Node2", "AdamicAdar"])
aa_df.sort_values("AdamicAdar", ascending=False).head(15)

## Preferential Attachment Score

Preferential attachment assumes that nodes with large degree are more likely to form new links.

This is based on the "rich get richer" intuition seen in many networks.

In [ ]:
pa_scores = list(nx.preferential_attachment(G, missing_edges))
pa_df = pd.DataFrame(pa_scores, columns=["Node1", "Node2", "PreferentialAttachment"])
pa_df.sort_values("PreferentialAttachment", ascending=False).head(15)

## Combine Multiple Link Prediction Scores

In practice, it is useful to place several heuristics side by side and compare their rankings.

In [ ]:
score_df = cn_df.merge(jaccard_df, on=["Node1", "Node2"])
score_df = score_df.merge(aa_df, on=["Node1", "Node2"])
score_df = score_df.merge(pa_df, on=["Node1", "Node2"])

score_df = score_df.sort_values(
    by=["CommonNeighbors", "AdamicAdar", "Jaccard"],
    ascending=False
).reset_index(drop=True)

score_df.head(20)

## Add an Embedding-Based Similarity Score

We can also score candidate links using **cosine similarity in embedding space**.  
This connects the embedding idea directly to link prediction.

In [ ]:
emb_scores = []
for u, v in missing_edges:
    i, j = nodes.index(u), nodes.index(v)
    emb_scores.append((u, v, sim_matrix[i, j]))

emb_df_scores = pd.DataFrame(emb_scores, columns=["Node1", "Node2", "EmbeddingCosine"])
combined_df = score_df.merge(emb_df_scores, on=["Node1", "Node2"])

combined_df.sort_values("EmbeddingCosine", ascending=False).head(20)

## Rank Candidate Links

We now rank candidate links using a mix of structural heuristics and embedding similarity.  
This gives a recommendation-style output for which missing edges look most plausible.

In [ ]:
ranked_df = combined_df.copy()
ranked_df["CN_rank"] = ranked_df["CommonNeighbors"].rank(ascending=False, method="min")
ranked_df["AA_rank"] = ranked_df["AdamicAdar"].rank(ascending=False, method="min")
ranked_df["Emb_rank"] = ranked_df["EmbeddingCosine"].rank(ascending=False, method="min")
ranked_df["AverageRank"] = ranked_df[["CN_rank", "AA_rank", "Emb_rank"]].mean(axis=1)

ranked_df = ranked_df.sort_values("AverageRank")
ranked_df.head(15)

## Visual Interpretation of a Top Predicted Link

Let us inspect the highest-ranked predicted pair.  
If the top candidate shares local neighbors or occupies a similar structural position, the recommendation will usually make intuitive sense.

In [ ]:
top_pair = ranked_df.iloc[0][["Node1", "Node2"]].tolist()
u, v = top_pair

print("Top predicted missing link:", (u, v))
print("Neighbors of", u, ":", sorted(G.neighbors(u)))
print("Neighbors of", v, ":", sorted(G.neighbors(v)))
print("Common neighbors:", sorted(nx.common_neighbors(G, u, v)))

## Simple Hold-Out Evaluation Setup

A more realistic evaluation removes some true edges from the graph, treats them as hidden positives, and checks whether the scoring method can recover them.

Here we create a simple train-test split over edges:
- keep most edges for the observed training graph
- hide a subset of true edges as test positives

In [ ]:
np.random.seed(42)

edges_list = list(G.edges())
train_edges, test_edges = train_test_split(edges_list, test_size=0.25, random_state=42)

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

print("Training edges:", len(train_edges))
print("Hidden test edges:", len(test_edges))
print("Training graph connected?:", nx.is_connected(G_train))

## Score Hidden Edges vs Random Non-Edges

For a lightweight classroom evaluation, we compare scores of:
- hidden true edges
- randomly selected non-edges

A good predictor should assign higher scores to hidden true edges on average.

In [ ]:
non_edges_train = list(nx.non_edges(G_train))
rng = np.random.default_rng(42)
sampled_non_edges = [non_edges_train[i] for i in rng.choice(len(non_edges_train), size=len(test_edges), replace=False)]

def score_pairs_adamic(graph, pairs):
    vals = list(nx.adamic_adar_index(graph, pairs))
    return pd.DataFrame(vals, columns=["Node1", "Node2", "Score"])

hidden_df = score_pairs_adamic(G_train, test_edges)
hidden_df["Label"] = 1

random_df = score_pairs_adamic(G_train, sampled_non_edges)
random_df["Label"] = 0

eval_df = pd.concat([hidden_df, random_df], ignore_index=True)
eval_df.head()

## Compare Average Scores for Positives and Negatives

This simple comparison is not a full benchmark, but it provides intuition for whether the heuristic is useful.

In [ ]:
summary_eval = eval_df.groupby("Label")["Score"].agg(["mean", "median", "max", "min", "count"]).reset_index()
summary_eval["Label"] = summary_eval["Label"].map({1: "Hidden true edges", 0: "Random non-edges"})
summary_eval

## Optional Visualization of Score Distributions

If hidden edges tend to have larger scores than random non-edges, the distributions should separate at least partially.

In [ ]:
plt.figure(figsize=(8, 5))
for label, grp in eval_df.groupby("Label"):
    plt.hist(grp["Score"], bins=10, alpha=0.6, label="Hidden true edges" if label == 1 else "Random non-edges")

plt.title("Adamic–Adar Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## Discussion: How Embeddings and Link Prediction Connect

Graph embeddings and link prediction are closely related.

- embeddings provide vector representations of nodes
- node-pair similarity in embedding space can act as a link score
- topological heuristics provide interpretable structural baselines
- learned embeddings often improve scalability and downstream ML integration

In advanced systems, embeddings can be learned specifically to improve link prediction performance.

# See you next lecture!